# 第 4 周练习 —— 交易代码生成器（多模型比较）

## 练习目标

本笔记本做一个由 **LLM** 驱动的**交易代码生成器**：

- 根据自然语言策略描述，生成可在**模拟环境**里买卖股票的 Python 代码
- 并排比较多个模型（GPT、Claude、Gemini、Grok 等）
- 按绩效排名（利润百分比、最终资产、成交笔数）
- 用 **Gradio UI** 做交互式对比

## 和本课 Week 4 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型 / OpenAI 兼容客户端 | 同一套 `chat.completions` 接口对接多家 API |
| 代码生成 + 沙箱执行 | `exec` 跑模型产出的策略代码 |
| Gradio 界面 | 生成、回测、排行榜一站完成 |

## 怎么跑

1. 从上到下依次运行单元格
2. 在项目根目录 `.env` 里准备要用的 API Key
3. 打开 Gradio，输入策略描述，点「Generate & Run」或「Compare All Models」


### 环境准备（Setup）

请确保项目根目录的 `.env` 里包含你要用的模型对应密钥（Environment Variables）：

- `OPENAI_API_KEY`、`ANTHROPIC_API_KEY`、`GOOGLE_API_KEY`、`GROK_API_KEY`、`GROQ_API_KEY`、`OPENROUTER_API_KEY`
- 本地模型：先用 `ollama run llama3.2`（或 `qwen2.5-coder`）把 Ollama 跑起来

没有云端 Key 时，后面会自动回退到本地 Ollama。


In [ ]:
# ========== 导入：生成代码、回测、Gradio 界面都要用到 ==========
# 标准库 os：读环境变量（API Key 等）
import os
# 标准库 traceback：沙箱执行出错时打印完整堆栈
import traceback
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量，避免写死在代码里
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：OpenAI 兼容 Chat Completions 客户端
from openai import OpenAI
# 导入 gradio：快速搭 Web UI，方便对比多个模型
import gradio as gr
# 导入 numpy：用随机游走生成模拟股价序列
import numpy as np


In [ ]:
# ========== 读密钥 + 初始化各家 OpenAI 兼容客户端 ==========
# 加载 .env；override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)
# 逐个读取各厂商 API Key（没有则为 None）
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# ---------- 连接客户端（OpenAI 兼容 base_url）----------
# 官方 OpenAI：默认读 OPENAI_API_KEY
# 连接到客户端库
# Anthropic OpenAI 兼容网关地址（字符串勿改）
openai_client = OpenAI()
# Google Gemini 的 OpenAI 兼容端点
anthropic_url = "https://api.anthropic.com/v1/"
# xAI Grok 端点
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# Groq 高速推理端点
grok_url = "https://api.x.ai/v1"
# 本地 Ollama OpenAI 兼容端口
groq_url = "https://api.groq.com/openai/v1"
# OpenRouter 聚合网关
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"
# 有 Key 才建客户端，否则置 None

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url) if anthropic_api_key else None
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url) if google_api_key else None
grok = OpenAI(api_key=grok_api_key, base_url=grok_url) if grok_api_key else None
# Ollama 本地常用占位 api_key="ollama"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url) if groq_api_key else None
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None
# 提示：客户端对象已就绪（真正可用模型看下一格）

print("API clients initialized.")


In [ ]:
# ========== 模型清单与 client 映射：有 Key 才挂载对应模型 ==========
# 根据已有 API 密钥定制；缺 Key 的厂商整段跳过
# 需要更多开源模型时，优先走 openrouter
# 先准备空列表/字典，后面按 Key 往里填
models_config = []
clients = {}

# OpenAI：挂载 gpt-4o / gpt-4o-mini
if openai_api_key:
    models_config.extend(["gpt-4o", "gpt-4o-mini"])
    clients.update({"gpt-4o": openai_client, "gpt-4o-mini": openai_client})
# Anthropic Claude：模型 id 必须与接口一致
if anthropic_api_key:
    models_config.extend(["claude-sonnet-4-20250514", "claude-3-5-haiku-20241022"])
    clients.update({"claude-sonnet-4-20250514": anthropic, "claude-3-5-haiku-20241022": anthropic})
# Google Gemini
if google_api_key:
    models_config.extend(["gemini-2.0-flash", "gemini-1.5-pro"])
    clients.update({"gemini-2.0-flash": gemini, "gemini-1.5-pro": gemini})
# xAI Grok
if grok_api_key:
    models_config.extend(["grok-3"])
    clients.update({"grok-3": grok})
# Groq 上的 Llama
if groq_api_key:
    models_config.extend(["llama-3.3-70b-versatile"])
    clients.update({"llama-3.3-70b-versatile": groq})
# OpenRouter：用 "厂商/模型" 形式的路由 id
if openrouter_api_key:
    models_config.extend(["openai/gpt-4o-mini", "anthropic/claude-3.5-haiku"])
    clients.update({"openai/gpt-4o-mini": openrouter, "anthropic/claude-3.5-haiku": openrouter})

# 若一个云端模型都没有，就只用本地 Ollama
# 后备：如果没有 API，则使用 Ollama（本地）
if not models_config:
    models_config = ["llama3.2", "qwen2.5-coder"]
    clients = {"llama3.2": ollama, "qwen2.5-coder": ollama}
else:
    # 有云端模型时，仍把本地模型追加进列表，方便对比
    models_config.extend(["llama3.2", "qwen2.5-coder"])
    clients.update({"llama3.2": ollama, "qwen2.5-coder": ollama})

# 打印当前可用模型名列表
print("Available models:", models_config)


## 模拟交易环境（Trading Simulator）

这里实现一个简单的**回测器**：提供价格序列，以及 `buy` / `sell` 接口。

LLM 生成的策略代码会在沙箱里执行，并拿到名为 `sim` 的模拟器对象。


In [ ]:
# ========== 价格序列生成 + TradingSimulator 回测沙箱 ==========
def generate_price_series(seed=42, days=100, start_price=100.0, volatility=0.02):
    """Generate realistic-looking price series (random walk with drift)."""
    # 固定随机种子，保证每次回测同一条价格路径，便于公平比模型
    np.random.seed(seed)
    # 日收益率 ~ 正态(小漂移, 波动率)，模拟带漂移的随机游走
    returns = np.random.normal(0.0002, volatility, days)
    # 把收益率累乘成价格路径：P_t = P0 * exp(cumsum(r))
    prices = start_price * np.exp(np.cumsum(returns))
    # 转成普通 list，方便后面给策略代码用
    return list(prices)

# 模拟交易环境：按日收盘价买卖
class TradingSimulator:
    """Simulated trading env: buy/sell at daily closes. Code uses sim.buy(day, qty), sim.sell(day, qty)."""
    # 初始化：价格、现金、持仓、成交记录
    def __init__(self, prices, initial_cash=100_000.0):
        # 日收盘价序列（index = 交易日）
        self.prices = prices
        # 现金余额
        self.cash = initial_cash
        # 持仓股数
        self.shares = 0.0
        # 成交记录：(day, buy/sell, qty, price)
        self.trades = []  # (day, "buy"/"sell", qty, price)
        # 记下初始资金，后面算收益率用
        self.initial_cash = initial_cash

    # 买入：校验日期与数量，现金足够才成交
    def buy(self, day, shares):
        # 非法日期或非正数量：直接忽略（策略代码容错）
        if day < 0 or day >= len(self.prices) or shares <= 0:
            return
        # 按当日收盘价成交
        price = self.prices[day]
        # 所需资金
        cost = shares * price
        # 现金够才买得动
        if cost <= self.cash:
            self.cash -= cost
            self.shares += shares
            self.trades.append((day, "buy", shares, price))

    # 卖出：不能卖超过当前持仓
    def sell(self, day, shares):
        # 非法日期或非正数量：直接忽略（策略代码容错）
        if day < 0 or day >= len(self.prices) or shares <= 0:
            return
        # 不能卖超过当前持仓
        sell_qty = min(shares, self.shares)
        if sell_qty > 0:
            price = self.prices[day]
            self.cash += sell_qty * price
            self.shares -= sell_qty
            self.trades.append((day, "sell", sell_qty, price))

    # 组合净值 = 现金 + 持股市值
    def portfolio_value(self, day=None):
        day = day if day is not None else len(self.prices) - 1
        day = min(day, len(self.prices) - 1)
        return self.cash + self.shares * self.prices[day]

    # 回测结束时的组合净值
    def final_value(self):
        return self.portfolio_value(len(self.prices) - 1)

# 评估默认价格系列
# 评估用的默认价格序列（固定 seed=42，100 个交易日）
DEFAULT_PRICES = generate_price_series(seed=42, days=100)


## 代码生成提示词（Prompt）

LLM 收到策略描述后，必须输出**只使用 `sim` 对象**的 Python 代码（不要 Markdown 解释）。

下面的 `SYSTEM_PROMPT` / `user_prompt` 字符串保持英文原文，改了会影响模型行为。


In [ ]:
# ========== System / User Prompt：约束模型只输出可执行交易代码 ==========
# SYSTEM_PROMPT / user_prompt 全文保持英文原文（影响模型行为，禁止翻译）
SYSTEM_PROMPT = """You are a trading algorithm developer. Your task is to write Python code that implements a trading strategy in a simulated environment.

You have access to a variable `sim` which is a TradingSimulator with:
- sim.prices: list of daily closing prices (index 0 = day 0, 1 = day 1, ...)
- sim.buy(day, shares): buy `shares` at the closing price of `day`
- sim.sell(day, shares): sell `shares` at the closing price of `day`
- sim.cash: current cash balance
- sim.shares: current shares held
- sim.portfolio_value(day): cash + shares * price at `day`

Your code will be executed with `sim` already created. Loop through days (0 to len(sim.prices)-1) and call sim.buy() or sim.sell() based on your strategy.
Do NOT redefine sim. Do NOT use print statements. Respond ONLY with valid Python code, no markdown or explanation."""

# 把用户的策略描述嵌进固定英文模板（字符串勿改）
def user_prompt(strategy_description):
    return f"""Write Python trading code for this strategy:

{strategy_description}

Use the `sim` object (TradingSimulator) provided. Iterate over days 0 to len(sim.prices)-1 and call sim.buy(day, qty) or sim.sell(day, qty).
Respond ONLY with Python code."""


In [ ]:
# ========== 调用指定模型生成策略代码 ==========
def generate_code(model, strategy_description):
    """Generate trading code using the specified model."""
    # 模型未配置或 client 为 None：直接返回错误信息
    if model not in clients or clients[model] is None:
        return None, f"Model {model} not available"
    # 取出该模型对应的 OpenAI 兼容客户端
    client = clients[model]
    # 组装 system + user 两条消息
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt(strategy_description)}
    ]
    try:
        # Chat Completions：让模型写策略代码
        response = client.chat.completions.create(model=model, messages=messages)
        # 取出助手回复文本
        code = response.choices[0].message.content
        # 剥离 Markdown 代码块
        # 剥离 Markdown 代码块围栏，留下纯 Python
        for delim in ["```python", "```"]:
            code = code.replace(delim, "")
        code = code.strip()
        # 成功：返回代码，错误位为 None
        return code, None
    # 网络/鉴权/模型错误：返回异常字符串
    except Exception as e:
        return None, str(e)


In [ ]:
# ========== 沙箱执行生成代码并汇总绩效指标 ==========
def run_strategy(code, prices=None):
    """Execute generated code in sandbox. Returns dict with profit_pct, final_value, error, trades, sharpe (approx)."""
    # 未传入价格序列时，用默认 DEFAULT_PRICES
    prices = prices or DEFAULT_PRICES
    # 为这次回测新建独立模拟器（避免状态串扰）
    sim = TradingSimulator(prices)
    # 白名单内置函数：限制策略代码能力，降低风险
    safe_builtins = {
        "len": len, "range": range, "enumerate": enumerate, "zip": zip,
        "min": min, "max": max, "sum": sum, "abs": abs, "round": round,
        "list": list, "float": float, "int": int, "True": True, "False": False,
    }
    # 命名空间里只注入 sim + 安全 builtins
    ns = {"sim": sim, "__builtins__": safe_builtins}
    try:
        # 在沙箱里执行模型生成的代码（会调用 sim.buy / sim.sell）
        exec(code, ns)
        # 计算期末净值
        final = sim.final_value()
        # 收益率（百分比）
        ret = (final - sim.initial_cash) / sim.initial_cash * 100
        # 简单的 Sharpe 代理：如果我们有的话，则返回 / (1 + 标准日收益)
        # 成交笔数
        n_trades = len(sim.trades)
        return {
            "final_value": final,
            "profit_pct": ret,
            "trades": n_trades,
            "error": None,
            "success": True,
        }
    # 运行失败：返回 0 收益 + 完整 traceback，便于 UI 展示
    except Exception as e:
        return {
            "final_value": sim.initial_cash,
            "profit_pct": 0,
            "trades": 0,
            "error": traceback.format_exc(),
            "success": False,
        }


In [ ]:
# ========== 对所有可用模型：生成代码 → 回测 → 按利润排序 ==========
def evaluate_all_models(strategy_description, models_to_run=None):
    """Generate code with each model, run simulation, return ranked results."""
    # 默认跑 models_config 里所有已绑定 client 的模型
    models_to_run = models_to_run or [m for m in models_config if m in clients and clients[m] is not None]
    # 收集每个模型的回测结果
    results = []
    # 逐个模型生成并评估
    for model in models_to_run:
        # 先让该模型生成策略代码
        code, err = generate_code(model, strategy_description)
        # 生成失败：用极低利润占位，方便排到后面
        if err:
            results.append({"model": model, "profit_pct": -999, "error": err, "success": False, "code": ""})
            continue
        # 生成成功：进沙箱回测
        r = run_strategy(code)
        r["model"] = model
        r["code"] = code
        results.append(r)
    # 按利润百分比排名（降序）
    # 按利润百分比排名（降序：最高利润第一）
    results.sort(key=lambda x: x["profit_pct"], reverse=True)
    # 返回已排序的结果列表
    return results


In [ ]:
# ========== 快速自检：不经过 LLM，手动买持验证模拟器 ==========
# 快速测试：手动运行简单的买入持有策略
# 用默认价格序列新建模拟器
sim = TradingSimulator(DEFAULT_PRICES)
# 交易日长度
n = len(sim.prices)
# 第 0 天买入 500 股，最后一天卖出
sim.buy(0, 500)
sim.sell(n - 1, 500)
# 打印期末净值与相对初始 10 万美元的涨跌幅
print(f"Buy-hold test: Initial $100k -> Final ${sim.final_value():,.0f} ({sim.final_value()/100000*100-100:.1f}%)")


## Gradio 用户界面

交互流程：

1. 输入策略描述（自然语言）
2. 选择模型并生成代码，立即回测
3. 查看生成代码与绩效结果
4. **比较所有模型** — 跑完整列表并看排行榜（利润最高者获胜）


In [ ]:
# ========== 排行榜 CSS：金银铜高亮前三名 ==========
# 排行榜的样式（给 Gradio Blocks 的 css= 参数用）
CSS = """
.leaderboard-box { font-family: monospace; padding: 16px; background: #1a1d23; border-radius: 10px; }
.rank-1 { color: #ffd700 !important; font-weight: bold; font-size: 1.1em; }
.rank-2 { color: #c0c0c0 !important; }
.rank-3 { color: #cd7f32 !important; }
"""


In [ ]:
# ========== 默认策略文案 + 排行榜 Markdown 格式化 ==========
# DEFAULT_STRATEGY 英文原文会进入 user prompt，禁止翻译
DEFAULT_STRATEGY = """Buy and hold: invest 50% of cash on day 0, then sell everything on the last day."""

# 把 evaluate_all_models 的结果格式化成 Markdown 排行榜
def format_leaderboard(results):
    # 标题行（展示字符串保持原文）
    lines = ["## 🏆 Model Leaderboard (ranked by profit %)\n"]
    # 按名次遍历结果
    for i, r in enumerate(results, 1):
        # 前三名用奖牌 emoji，其余用名次数字
        medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}."
        # 失败条目附加截断后的错误信息
        err = f" — Error: {r.get('error', '')[:80]}..." if not r.get("success", True) else ""
        profit = r.get("profit_pct", 0)
        trades = r.get("trades", 0)
        # 一行：名次 + 模型名 + 利润 + 成交笔数
        lines.append(f"**{medal} {r['model']}** — Profit: {profit:.2f}% | Trades: {trades}{err}")
    # 拼成完整 Markdown 文本
    return "\n".join(lines)


In [ ]:
# ========== Gradio 回调：单模型生成 / 全模型对比 ==========
# 单模型：生成代码并立刻回测
def ui_generate(model, strategy):
    # 用下拉框选中的模型生成代码
    code, err = generate_code(model, strategy)
    # 生成失败：把错误信息返回到 Result 文本框
    if err:
        return code or "", f"Error: {err}"
    # 生成成功：立刻在沙箱回测
    r = run_strategy(code)
    # 运行期异常：截断堆栈，避免 UI 被刷屏
    if r["error"]:
        return code, f"Runtime error:\n{r['error'][:500]}"
    # 成功：返回代码 + 一行绩效摘要
    return code, f"Profit: {r['profit_pct']:.2f}% | Trades: {r['trades']} | Final: ${r['final_value']:,.0f}"

# 全模型对比：生成、回测、排行榜
def ui_compare_all(strategy):
    # 对全部可用模型生成并回测，得到已排序结果
    results = evaluate_all_models(strategy)
    # 第一名（利润最高）
    best = results[0] if results else None
    # 冠军模型生成的代码
    best_code = best.get("code", "") if best else ""
    # 格式化为 Markdown 排行榜
    leaderboard = format_leaderboard(results)
    # 胜出模型名（需 success）
    winner = best["model"] if best and best.get("success") else "None"
    # 摘要文案
    msg = f"Best model: {winner} (profit: {best.get('profit_pct', 0):.2f}%)" if best else "No results"
    # 输出：冠军代码、排行榜、摘要文案
    return best_code, leaderboard, msg


In [ ]:
# ========== 组装 Gradio Blocks：输入策略 → 生成/对比 → 展示结果 ==========
# title / Markdown / label 等展示字符串保持原文
# 创建 Gradio Blocks 应用
with gr.Blocks(css=CSS, title="Trading Code Generator — Multi-Model Comparison") as ui:
    # 页面标题
    gr.Markdown("# 📈 Trading Code Generator — Compare LLM Models")
    # 页面简介
    gr.Markdown("Generate trading strategies with different models and see which produces the best code.")

    with gr.Row():
        with gr.Column(scale=1):
            # 策略描述输入框（默认填 DEFAULT_STRATEGY）
            strategy = gr.Textbox(
                label="Strategy description",
                value=DEFAULT_STRATEGY,
                placeholder="e.g., Buy when price drops 2% from the previous day, sell when it rises 3%.",
                lines=4
            )
            with gr.Row():
                # 模型下拉：来自前面组装的 models_config
                model_dropdown = gr.Dropdown(models_config, value=models_config[0], label="Model")
                # 单模型：生成并回测
                gen_btn = gr.Button("Generate & Run", variant="primary")
            with gr.Row():
                # 全模型对比按钮
                compare_btn = gr.Button("🏆 Compare All Models", variant="secondary")

        with gr.Column(scale=1):
            # 展示生成的 Python 策略代码
            code_out = gr.Code(label="Generated code", language="python", lines=15)
            # 单行/短文本结果摘要
            result_out = gr.Textbox(label="Result", lines=3)
            # Markdown 排行榜区域
            leaderboard_out = gr.Markdown(label="Leaderboard")

    # 绑定：Generate & Run → ui_generate
    gen_btn.click(
        fn=ui_generate,
        inputs=[model_dropdown, strategy],
        outputs=[code_out, result_out]
    )
    # 绑定：Compare All → ui_compare_all
    compare_btn.click(
        fn=ui_compare_all,
        inputs=[strategy],
        outputs=[code_out, leaderboard_out, result_out]
    )

# 启动 Gradio；inbrowser=True 会尝试自动打开浏览器
ui.launch(inbrowser=True)
